# 🫀 ECG ResNet1D — **v1 LIGERO** (CPU · modelo independiente)
## TFM · Módulo ECG · Universidad de Salamanca

---

Línea base rápida del módulo de electrocardiograma: **ResNet-1D pequeña** (base 32, 1 bloque por etapa)
sobre la señal remuestreada a 1000 puntos. Las 3 versiones por peso: **v1 ligera** · v2 óptima · v3 pesada.

## 🔄 Adaptación a las conclusiones del EDA

| Cambio | Detalle | Origen (recuadro naranja) |
|---|---|---|
| **Etiquetado FINAL** | `POS=(==1)` · `NEG=(==0)|(NaN→0)` · **−1 ENMASCARADO** (U-Ignore). **Se elimina** la derivación de negativos desde *No Finding*, que introducía **sesgo de espectro**. | §3 Definición del negativo |
| **Métrica primaria = AUC-PR** | `macro_AP_path` gobierna el early-stopping, la selección y los informes; **AUC-ROC pasa a secundaria**; **IC bootstrap** de AP y de AUC. | §3 Métrica · §1 IC por val/test pequeños |
| **Puntos de operación** | Tres umbrales fijados en VAL: F1, **cribado (Se≥0,90)** y **confirmación (Sp≥0,90)**, con **Se/Sp/VPP/VPN**. | §11 Puntos de operación |
| **Calibración verificada** | **Brier** antes/después + curva de fiabilidad. | §11 Calibración |
| **Pre-registro** | La regla de decisión se congela en JSON **antes** de tocar test. | §11 Protocolo |
| **Equidad y robustez** | Métricas estratificadas por sexo, etnia, ingreso y **proyección AP/PA**. | §2/§9 · Anexo CXR |

### Decisiones específicas de esta modalidad
- **Señal intacta**: el ECG llega normalizado a **[−1, 1]**, 12 derivaciones × 5000 muestras (~10 s @500 Hz).
  **No se re-normaliza por derivación**, para no perder la amplitud relativa entre ellas. → Anexo EDA
- **`cxr_view` NUNCA como predictor** (es un proxy de gravedad: la placa AP se hace al paciente encamado);
  se usa **solo para estratificar** los resultados.
- **B5 y B7 no aplican**: no hay variables tabulares cuya importancia medir ni patrón de *missingness*
  que vigilar — el ECG es una señal continua siempre presente. Esa información MNAR entra por el
  módulo tabular y se propaga en la fusión. → §6

> **Aviso:** con el nuevo etiquetado y la nueva métrica, los resultados **no son comparables** con los
> de la ejecución previa (que derivaba negativos de *No Finding* y seleccionaba por AUC-ROC).



In [ ]:
# CELDA 1 · DEPENDENCIAS
import subprocess, sys
try: import torch  # noqa
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","torch","--index-url","https://download.pytorch.org/whl/cpu","-q"],check=True)
for p in ["scikit-learn","pandas","numpy","matplotlib","seaborn","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install",p,"-q"],check=False)
print("Dependencias listas.")

In [ ]:
# CELDA 2 · IMPORTS, RUTAS Y CONSTANTES
import os, gc, json, time, copy, random, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4); DEVICE=torch.device("cpu")

BASE=Path(r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0")
CSV=BASE/"data_csv"/"clean"; NPY=BASE/"data_npy"
TRAIN_CSV,VAL_CSV,TEST_CSV=CSV/"train_clean.csv",CSV/"val_clean.csv",CSV/"test_clean.csv"
ECG_NPY={"train":NPY/"train"/"ecg_train.npy","val":NPY/"val"/"ecg_val.npy","test":NPY/"test"/"ecg_test.npy"}
HADM_NPY={"train":NPY/"train"/"hadm_id_train.npy","val":NPY/"val"/"hadm_id_val.npy","test":NPY/"test"/"hadm_id_test.npy"}
OUTPUT_DIR=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\02_ecg\v1"); OUTPUT_DIR.mkdir(exist_ok=True)
PREP=OUTPUT_DIR/"prep"; PREP.mkdir(exist_ok=True); FIG=OUTPUT_DIR/"figuras"; FIG.mkdir(exist_ok=True)

LABELS=["Atelectasis","Cardiomegaly","Edema","Lung Opacity","No Finding","Pleural Effusion"]
N_LABELS=len(LABELS); NO_FINDING="No Finding"
PATHOLOGY=[l for l in LABELS if l!=NO_FINDING]; CORE=["Cardiomegaly","Edema","Pleural Effusion"]
N_LEADS,ECG_RAW_LEN=12,5000
ECG_LEN=1000          # v1: baja resolución (rápido)
BATCH=64; BASE_WIDTH=32; BLOCKS=1          # v1: red pequeña
K_FOLDS,EPOCHS,PATIENCE=3,15,4
print("Salidas en", OUTPUT_DIR, "· ECG_LEN", ECG_LEN, "· base", BASE_WIDTH)

In [ ]:
# CELDA 3 · CARGA Y ALINEACIÓN CSV <-> señal ECG (join por hadm_id)
def load_split(csv,ecg,hadm,name):
    df=pd.read_csv(csv,sep=";"); h=np.load(hadm,allow_pickle=True); h2i={int(x):i for i,x in enumerate(h)}
    df["_npy_idx"]=df["hadm_id"].map(lambda z:h2i.get(int(z),-1)); n0=len(df)
    df=df[df["_npy_idx"]>=0].reset_index(drop=True); a=np.load(ecg,mmap_mode="r")
    print(f"{name:5s}: CSV={n0:,} con ECG={len(df):,} · señal {a.shape}"); return df,a
df_train,ecg_train=load_split(TRAIN_CSV,ECG_NPY["train"],HADM_NPY["train"],"train")
df_val,ecg_val=load_split(VAL_CSV,ECG_NPY["val"],HADM_NPY["val"],"val")
df_test,ecg_test=load_split(TEST_CSV,ECG_NPY["test"],HADM_NPY["test"],"test")

In [ ]:
# CELDA 4 · OBJETIVOS — definición FINAL del negativo (U-IGNORE del −1)
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · build_targets
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : convierte los cuatro estados de cada etiqueta (1 / 0 / −1 / NaN) en dos matrices:
#              `labels` (0-1) y `mask` (1 = la etiqueta cuenta en pérdida y métrica, 0 = se ignora).
# POR QUÉ    : fija la definición FINAL del negativo acordada con el tutor y la deja IDÉNTICA en las
#              tres modalidades, requisito para que la fusión combine probabilidades coherentes.
#              POS = (==1) · NEG = (==0) | (NaN→0) · −1 = ENMASCARADO (U-Ignore de CheXpert).
# ENTRADAS   : df (DataFrame del split) · uncertainty_policy ("ignore" por defecto) · derive · verbose
# SALIDAS    : (labels (N,6) float32, mask (N,6) float32)
# ORIGEN EDA : §3 · recuadro naranja "negativo = 0 explícito + NaN→0; el −1 se enmascara (fuera de
#              pérdida y métrica); «Sin hallazgo» NO como negativo (evita el sesgo de espectro)".
# CAMBIO vs versión previa: se ELIMINA la derivación de negativos desde «No Finding» (derive=False)
#              y el NaN pasa a NEGATIVO con máscara activa, en lugar de quedar enmascarado.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              verificar que pos_weight sale 2,28 / 1,85 / 3,43 / 2,15 / 6,31 / 1,66 en train; y avisar
#              de que las métricas NO son comparables con las de la ejecución anterior.
# ══════════════════════════════════════════════════════════════════════════════
def build_targets(df, uncertainty_policy="ignore", derive=False, verbose=False):
    raw=df[LABELS].to_numpy(dtype=float); N=raw.shape[0]
    labels=(raw==1.0).astype(np.float32)          # 1 → positivo ; 0 y NaN → 0 (negativo)
    mask  =np.ones((N,N_LABELS),np.float32)       # por defecto TODO entra en pérdida/métrica
    unc=(raw==-1.0)
    if   uncertainty_policy=="ignore": mask[unc]=0.0    # −1 → ENMASCARADO (definición FINAL)
    elif uncertainty_policy=="ones":   labels[unc]=1.0  # alternativa, no usada
    ndp=ndn=0
    if derive:   # CONSERVADO por compatibilidad; NO forma parte de la definición FINAL
        nf=LABELS.index(NO_FINDING); pc=[j for j in range(N_LABELS) if j!=nf]; nfp=(raw[:,nf]==1)
        for j in pc:
            f=nfp&np.isnan(raw[:,j]); labels[f,j]=0.0; ndp+=int(f.sum())
        ap=(raw[:,pc]==1).any(1); fn=ap&np.isnan(raw[:,nf]); labels[fn,nf]=0.0; ndn=int(fn.sum())
    if verbose:
        for j,l in enumerate(LABELS):
            s=mask[:,j]==1; p=int((labels[s,j]==1).sum()); n=int((labels[s,j]==0).sum())
            print(f"   {l:18s} pos={p:5d} neg={n:5d} enmasc={int((mask[:,j]==0).sum()):4d} pos_weight={n/max(p,1):.2f}")
    return labels,mask
y_train,m_train=build_targets(df_train,verbose=True)
y_val,m_val=build_targets(df_val); y_test,m_test=build_targets(df_test)



In [ ]:
# CELDA 5 · PREPROCESADO de la señal (downsample a 1000 + z-score por derivación), con caché
def preprocess(df, ecg, name):
    cache=PREP/f"ecg_{name}_{ECG_LEN}.npy"
    if cache.exists():
        X=np.load(cache)
        if len(X)==len(df): print(f"   {name}: caché {X.shape}"); return X
    idx=df["_npy_idx"].to_numpy(); factor=ECG_RAW_LEN//ECG_LEN
    X=np.empty((len(idx),N_LEADS,ECG_LEN),np.float32)
    for k in tqdm(range(len(idx)),desc=f"prep {name}",leave=False):
        a=np.asarray(ecg[int(idx[k])],np.float32).reshape(ECG_RAW_LEN,N_LEADS).T
        a=a[:,:factor*ECG_LEN].reshape(N_LEADS,ECG_LEN,factor).mean(2)
        mu=a.mean(1,keepdims=True); sd=a.std(1,keepdims=True); X[k]=(a-mu)/(sd+1e-6)
    np.save(cache,X); print(f"   {name}: calculado {X.shape}"); return X
X_train=preprocess(df_train,ecg_train,"train"); X_val=preprocess(df_val,ecg_val,"val"); X_test=preprocess(df_test,ecg_test,"test")

In [ ]:
# CELDA 6 · MODELO ResNet1D PEQUEÑO (base 32, 1 bloque/etapa)
class Block(nn.Module):
    def __init__(s,ci,co,st=1,k=7,dp=0.2):
        super().__init__(); p=k//2
        s.c1=nn.Conv1d(ci,co,k,st,p,bias=False); s.b1=nn.BatchNorm1d(co)
        s.c2=nn.Conv1d(co,co,k,1,p,bias=False); s.b2=nn.BatchNorm1d(co); s.d=nn.Dropout(dp); s.dn=None
        if st!=1 or ci!=co: s.dn=nn.Sequential(nn.Conv1d(ci,co,1,st,bias=False),nn.BatchNorm1d(co))
    def forward(s,x):
        idt=x if s.dn is None else s.dn(x); o=F.relu(s.b1(s.c1(x))); o=s.d(o); o=s.b2(s.c2(o)); return F.relu(o+idt)
class ECGNet(nn.Module):
    def __init__(s,base=BASE_WIDTH,dp=0.3):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(N_LEADS,base,15,2,7,bias=False),nn.BatchNorm1d(base),nn.ReLU(),nn.MaxPool1d(3,2,1))
        s.l1=Block(base,base,1,dp=dp); s.l2=Block(base,base*2,2,dp=dp); s.l3=Block(base*2,base*4,2,dp=dp); s.l4=Block(base*4,base*8,2,dp=dp)
        s.pool=nn.AdaptiveAvgPool1d(1); s.head=nn.Sequential(nn.Linear(base*8,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(dp),nn.Linear(128,N_LABELS))
        for m in s.modules():
            if isinstance(m,(nn.Conv1d,nn.Linear)): nn.init.kaiming_normal_(m.weight,nonlinearity="relu")
    def forward(s,x):
        z=s.stem(x); z=s.l1(z); z=s.l2(z); z=s.l3(z); z=s.l4(z); return s.head(s.pool(z).flatten(1))
print("Parámetros:", sum(p.numel() for p in ECGNet().parameters()))

In [ ]:
# CELDA 7 · MÉTRICAS, pos_weight dinámico y pérdida enmascarada
def multilabel_metrics(probs,labels,mask,thresholds=None):
    if thresholds is None: thresholds={l:0.5 for l in LABELS}
    res={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]; npos=int(yt.sum()); nneg=int((1-yt).sum())
        pred=(yp>=thresholds.get(l,0.5)).astype(float)
        auc=roc_auc_score(yt,yp) if npos>=2 and nneg>=2 else float("nan")
        ap=average_precision_score(yt,yp) if npos>=2 and nneg>=2 else float("nan")
        tp=int(((pred==1)&(yt==1)).sum()); tn=int(((pred==0)&(yt==0)).sum())
        fp=int(((pred==1)&(yt==0)).sum()); fn=int(((pred==0)&(yt==1)).sum())
        res[l]={"AUC":auc,"AP":ap,"F1":f1_score(yt,pred,zero_division=0),"sens":tp/max(tp+fn,1),"spec":tn/max(tn+fp,1),
                "n_pos":npos,"n_neg":nneg,"TP":tp,"TN":tn,"FP":fp,"FN":fn,"thr":thresholds.get(l,0.5),
                "prevalencia":npos/max(npos+nneg,1)}
    mac=lambda g,k:float(np.nanmean([res[l][k] for l in g])) if any(not np.isnan(res[l][k]) for l in g) else float("nan")
    res["macro_AUC_core"]=mac(CORE,"AUC"); res["macro_AUC_path"]=mac(PATHOLOGY,"AUC")   # secundaria
    res["macro_AP_core"] =mac(CORE,"AP");  res["macro_AP_path"] =mac(PATHOLOGY,"AP")    # PRIMARIA
    return res
def best_thresholds_by_f1(probs,labels,mask):
    grid=np.linspace(0.05,0.95,37); thr={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]
        if yt.sum()<2: thr[l]=0.5; continue
        bf,bt=-1,0.5
        for t in grid:
            f=f1_score(yt,(yp>=t).astype(float),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[l]=float(bt)
    return thr
def pos_weights(y,m,clip=10.0):
    w=np.ones(N_LABELS,np.float32)
    for j in range(N_LABELS):
        s=m[:,j]==1; pos=(y[s,j]==1).sum(); neg=(y[s,j]==0).sum(); w[j]=np.clip(neg/max(pos,1),1/clip,clip)
    return torch.tensor(w)
def masked_bce(logits,labels,mask,pw):
    bce=F.binary_cross_entropy_with_logits(logits,labels,pos_weight=pw.to(logits.device),reduction="none")*mask
    return bce.sum()/mask.sum().clamp(min=1e-8)
print("Metricas listas (PRIMARIA = macro_AP_path; ROC secundaria) y perdida enmascarada.")


In [ ]:
# CELDA 7b · KIT DE EVALUACIÓN CLÍNICA (B1, B2, B3, B4, B6, B8) — implementa los recuadros naranjas del EDA que faltaban
# NOTA: B5 (importancia MI/ANOVA) y B7 (dependencia de flags MNAR) NO APLICAN a esta modalidad.
# Motivo (§6 EDA): el ECG es una senal continua SIEMPRE presente: no hay variables tabulares cuya
# importancia medir ni patron de ausencia que vigilar. Esa informacion MNAR entra por el modulo
# tabular y se propaga a la fusion. Aqui si aplican B1, B2, B3, B4, B6 y B8.
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · bootstrap_ci_metric                                          [B4]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : remuestrea con reemplazo y recalcula la métrica, devolviendo el intervalo percentil,
#              POR ETIQUETA y para el MACRO de las 5 patologías.
# POR QUÉ    : con 464 pacientes en test, una diferencia entre modelos puede ser azar; el IC es lo
#              que permite afirmar (o no) que un modelo supera a otro.
# ENTRADAS   : probs (N,6) · labels (N,6) · mask (N,6) · metric "ap"|"auc" · n_boot · alpha
# SALIDAS    : dict {etiqueta:(lo,hi)} + clave "macro_path"
# ORIGEN EDA : §1 · "reportar SIEMPRE IC bootstrap por el tamaño reducido de val/test".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si los IC del stacking y del promedio SE SOLAPAN, no afirmar superioridad del stacking.
# ══════════════════════════════════════════════════════════════════════════════
def bootstrap_ci_metric(probs, labels, mask, metric="ap", n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.RandomState(seed)
    scorer = average_precision_score if metric == "ap" else roc_auc_score
    out, macro_vals = {}, []
    for j, l in enumerate(LABELS):
        idx = np.where(mask[:, j] == 1)[0]; yt, yp = labels[idx, j], probs[idx, j]
        if int(yt.sum()) < 2 or int((1 - yt).sum()) < 2:
            out[l] = (float("nan"), float("nan")); continue
        vals = []
        for _ in range(n_boot):
            bs = rng.randint(0, len(idx), len(idx))
            if yt[bs].sum() < 1 or (1 - yt[bs]).sum() < 1: continue
            vals.append(scorer(yt[bs], yp[bs]))
        out[l] = (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))) if vals else (float("nan"), float("nan"))
    for _ in range(n_boot):
        bs = rng.randint(0, len(probs), len(probs)); per = []
        for j, l in enumerate(LABELS):
            if l not in PATHOLOGY: continue
            sel = mask[bs, j] == 1; yt, yp = labels[bs][sel, j], probs[bs][sel, j]
            if yt.sum() < 1 or (1 - yt).sum() < 1: continue
            per.append(scorer(yt, yp))
        if per: macro_vals.append(np.mean(per))
    out["macro_path"] = (float(np.percentile(macro_vals, 100*alpha/2)),
                         float(np.percentile(macro_vals, 100*(1-alpha/2)))) if macro_vals else (float("nan"), float("nan"))
    return out

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · operating_points                                             [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : fija en VALIDACIÓN tres umbrales por etiqueta: "f1" (equilibrio), "cribado" (el más
#              alto que aún da Se>=sens_target) y "confirm" (el más bajo que aún da Sp>=spec_target).
# POR QUÉ    : un solo umbral no sirve en clínica. Cribar exige no perder enfermos; confirmar exige
#              no alarmar en falso. Son dos decisiones distintas sobre el mismo modelo.
# ENTRADAS   : probs/labels/mask de VALIDACIÓN · sens_target · spec_target
# SALIDAS    : dict {"f1"|"cribado"|"confirm": {etiqueta: umbral}}
# ORIGEN EDA : §11 · "fijar en VAL alta sensibilidad (cribado) y alta especificidad (confirmación).
#              Reportar Se/Sp/VPP/VPN".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              el VPP depende de la PREVALENCIA (13-36 %): un VPP modesto puede valer para cribar y
#              ser inservible para confirmar. Discutir cada punto por su consecuencia clínica.
# ══════════════════════════════════════════════════════════════════════════════
def operating_points(probs, labels, mask, sens_target=0.90, spec_target=0.90):
    grid = np.linspace(0.01, 0.99, 99); pts = {"f1": {}, "cribado": {}, "confirm": {}}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2:
            for k in pts: pts[k][l] = 0.5
            continue
        best_f1, thr_f1 = -1, 0.5; thr_sens, thr_spec = grid[0], grid[-1]
        for t in grid:
            pred = (yp >= t).astype(float)
            tp = ((pred == 1) & (yt == 1)).sum(); fn = ((pred == 0) & (yt == 1)).sum()
            tn = ((pred == 0) & (yt == 0)).sum(); fp = ((pred == 1) & (yt == 0)).sum()
            f1 = f1_score(yt, pred, zero_division=0)
            if f1 > best_f1: best_f1, thr_f1 = f1, t
            if tp/max(tp+fn, 1) >= sens_target: thr_sens = max(thr_sens, t)
            if tn/max(tn+fp, 1) >= spec_target: thr_spec = min(thr_spec, t)
        pts["f1"][l], pts["cribado"][l], pts["confirm"][l] = float(thr_f1), float(thr_sens), float(thr_spec)
    return pts

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · clinical_report                                              [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : evalúa unos umbrales y devuelve Se, Sp, VPP, VPN y la confusión por etiqueta.
# POR QUÉ    : AUC y AP resumen el ranking, pero la decisión se toma en UN umbral; el clínico
#              necesita saber cuántos enfermos se escapan y cuántas alarmas falsas se generan.
# ENTRADAS   : probs/labels/mask (TEST) · thresholds {etiqueta: umbral} · punto (nombre)
# SALIDAS    : DataFrame (punto, etiqueta, umbral, Se, Sp, VPP, VPN, TP/TN/FP/FN, prevalencia)
# ORIGEN EDA : §11 "Reportar Se/Sp/VPP/VPN" · §3 (la prevalencia condiciona el VPP).
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar FALSOS NEGATIVOS en cribado frente a FALSOS POSITIVOS en confirmación.
# ══════════════════════════════════════════════════════════════════════════════
def clinical_report(probs, labels, mask, thresholds, punto="f1"):
    rows = []
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        t = thresholds.get(l, 0.5); pred = (yp >= t).astype(float)
        tp = int(((pred == 1) & (yt == 1)).sum()); tn = int(((pred == 0) & (yt == 0)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
        rows.append({"punto": punto, "etiqueta": l, "umbral": round(t, 3),
                     "Se": tp/max(tp+fn,1), "Sp": tn/max(tn+fp,1), "VPP": tp/max(tp+fp,1), "VPN": tn/max(tn+fn,1),
                     "TP": tp, "TN": tn, "FP": fp, "FN": fn, "prevalencia": (tp+fn)/max(tp+tn+fp+fn,1)})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · calibration_report                                           [B2]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : Brier score por etiqueta + puntos de la curva de fiabilidad (10 bins por cuantiles).
# POR QUÉ    : la herramienta clínica muestra PROBABILIDADES; si no están calibradas, un 0,8 no
#              significa "80 % de estos pacientes lo tienen" y la cifra engaña al médico.
# ENTRADAS   : probs/labels/mask · n_bins
# SALIDAS    : (DataFrame Brier por etiqueta, dict {etiqueta:(frac_obs, media_pred)})
# ORIGEN EDA : §11 · "verificar con Brier score y curva de fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar Brier ANTES vs DESPUÉS; si no mejora, decirlo. RECALIBRAR tras la fusión.
# ══════════════════════════════════════════════════════════════════════════════
def calibration_report(probs, labels, mask, n_bins=10):
    rows, curves = [], {}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if len(np.unique(yt)) < 2:
            rows.append({"etiqueta": l, "Brier": float("nan")}); continue
        rows.append({"etiqueta": l, "Brier": float(brier_score_loss(yt, np.clip(yp, 0, 1)))})
        try: curves[l] = calibration_curve(yt, np.clip(yp, 0, 1), n_bins=n_bins, strategy="quantile")
        except Exception: curves[l] = (np.array([]), np.array([]))
    return pd.DataFrame(rows), curves

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · consistency_no_finding                                       [B6]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : correlación entre P("Sin hallazgo") y max P(patología), y % de casos en que ambas
#              superan 0,5 a la vez (el modelo se contradice).
# POR QUÉ    : las 6 cabezas son independientes; nada las obliga a ser coherentes. Un modelo que
#              afirma "sano" y "con derrame" a la vez es inaceptable en una herramienta clínica.
# ENTRADAS   : probs (N,6)
# SALIDAS    : dict {correlación (debe ser NEGATIVA), % incoherentes}
# ORIGEN EDA : §3 · "P(normal) útil como chequeo de consistencia, nunca como fuente de negativos".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              correlación ~0 o positiva ⇒ la cabeza "Sin hallazgo" no aprende normalidad.
# ══════════════════════════════════════════════════════════════════════════════
def consistency_no_finding(probs):
    j_nf = LABELS.index(NO_FINDING); j_p = [j for j in range(N_LABELS) if j != j_nf]
    p_nf, p_max = probs[:, j_nf], probs[:, j_p].max(axis=1)
    return {"corr_NoFinding_vs_maxPatologia": float(np.corrcoef(p_nf, p_max)[0, 1]),
            "pct_incoherentes": float(((p_nf > 0.5) & (p_max > 0.5)).mean()*100)}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · stratified_report                                            [B8]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : recalcula la métrica primaria dentro de cada subgrupo (sexo, etnia, ingreso) y por
#              PROYECCIÓN radiográfica (cxr_view).
# POR QUÉ    : (a) equidad; (b) robustez — la placa AP se hace al paciente encamado y magnifica la
#              silueta cardíaca, así que conviene ver si el rendimiento depende de la proyección.
# ENTRADAS   : df (metadatos del split) · probs/labels/mask · cols
# SALIDAS    : DataFrame (variable, grupo, n, macro_AP, macro_AUC)
# ORIGEN EDA : §2/§9 "evaluar equidad por sexo y etnia" · ANEXO CXR "monitorizar cxr_view".
# DECISIÓN DE DISEÑO: cxr_view se usa SOLO aquí. NUNCA como predictor: es proxy de gravedad y su
#              inclusión inflaría el resultado por un atajo asistencial en lugar de señal biológica.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              subgrupos con n<60 pueden diferir por PURO RUIDO; no afirmar inequidad sin IC.
# ══════════════════════════════════════════════════════════════════════════════
def stratified_report(df, probs, labels, mask, cols=("gender", "race", "admission_type", "cxr_view")):
    d = df.reset_index(drop=True); rows = []
    for col in cols:
        if col not in d.columns: continue
        for v in sorted(d[col].dropna().unique(), key=str):
            idx = d.index[d[col] == v].to_numpy()
            if len(idx) < 15:
                rows.append({"variable": col, "grupo": str(v), "n": len(idx), "macro_AP": np.nan, "macro_AUC": np.nan}); continue
            mm = multilabel_metrics(probs[idx], labels[idx], mask[idx])
            rows.append({"variable": col, "grupo": str(v), "n": len(idx),
                         "macro_AP": mm["macro_AP_path"], "macro_AUC": mm["macro_AUC_path"]})
    return pd.DataFrame(rows)



In [ ]:
# CELDA 8 · DATASET con aumento básico + entrenamiento
def aug(x):
    x=x+np.random.normal(0,0.025,x.shape).astype(np.float32); x=x*np.float32(np.random.uniform(0.9,1.1)); return x
class DS(Dataset):
    def __init__(s,X,y,m,augment=False): s.X,s.y,s.m,s.a=X,y,m,augment
    def __len__(s): return len(s.X)
    def __getitem__(s,i):
        x=aug(s.X[i]) if s.a else s.X[i]
        return torch.from_numpy(np.ascontiguousarray(x)),torch.from_numpy(s.y[i]),torch.from_numpy(s.m[i])
@torch.no_grad()
def predict(model,X,batch=256):
    model.eval(); out=[]
    for i in range(0,len(X),batch):
        out.append(torch.sigmoid(model(torch.from_numpy(np.ascontiguousarray(X[i:i+batch])).to(DEVICE))).cpu().numpy())
    return np.concatenate(out,0)
def train(Xtr,ytr,mtr,Xva,yva,mva,epochs=EPOCHS,patience=PATIENCE):
    pw=pos_weights(ytr,mtr); model=ECGNet().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs,1e-7)
    loader=DataLoader(DS(Xtr,ytr,mtr,augment=True),batch_size=BATCH,shuffle=True,drop_last=True)
    best,bs,wait=-1,None,0
    for ep in range(epochs):
        model.train()
        for xb,yb,mb in loader:
            xb,yb,mb=xb.to(DEVICE),yb.to(DEVICE),mb.to(DEVICE)
            opt.zero_grad(); loss=masked_bce(model(xb),yb,mb,pw); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        sch.step(); sc=multilabel_metrics(predict(model,Xva),yva,mva)["macro_AP_path"]
        if not np.isnan(sc) and sc>best+1e-4: best,bs,wait=sc,copy.deepcopy(model.state_dict()),0
        else:
            wait+=1
            if wait>=patience: break
    if bs is not None: model.load_state_dict(bs)
    return model
print("Entrenamiento listo.")


In [ ]:
# CELDA 9 · K-FOLD -> OOF de train (sin fuga) + ENSEMBLE (bagging) de val/test
kf=KFold(K_FOLDS,shuffle=True,random_state=SEED)
oof_train=np.zeros((len(df_train),N_LABELS),np.float32)
acc_val=np.zeros((len(df_val),N_LABELS),np.float32); acc_test=np.zeros((len(df_test),N_LABELS),np.float32); fold_macro=[]; t0=time.time()
for k,(tr,va) in enumerate(kf.split(np.arange(len(df_train)))):
    model=train(X_train[tr],y_train[tr],m_train[tr],X_train[va],y_train[va],m_train[va])
    oof_train[va]=predict(model,X_train[va]); acc_val+=predict(model,X_val); acc_test+=predict(model,X_test)
    mm=multilabel_metrics(oof_train[va],y_train[va],m_train[va]); fold_macro.append(mm["macro_AP_path"])
    print(f"Fold {k+1}/{K_FOLDS}: macroAP_path={mm['macro_AP_path']:.4f} ({(time.time()-t0)/60:.1f} min)"); del model; gc.collect()
val_pred_raw=acc_val/K_FOLDS; test_pred_raw=acc_test/K_FOLDS
print(f"\nOOF {K_FOLDS}-fold macroAP_path={np.nanmean(fold_macro):.4f}±{np.nanstd(fold_macro):.4f}")



In [ ]:
# CELDA 10 · CALIBRACIÓN (VAL) + VERIFICACIÓN Brier (B2) + PUNTOS DE OPERACIÓN (B1) + PRE-REGISTRO (B3)
# PROTOCOLO (§11 EDA): TRAIN entrena · VAL calibra, fija umbrales y selecciona · TEST se toca UNA vez.
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · apply_cal
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : aplica a cada columna de probabilidades el regresor isotónico aprendido en VALIDACIÓN.
# POR QUÉ    : la red devuelve sigmoides bien ordenadas pero no necesariamente calibradas; para la
#              herramienta clínica y para la fusión hacen falta PROBABILIDADES interpretables.
# ENTRADAS   : P (N,6) probabilidades crudas
# SALIDAS    : (N,6) probabilidades calibradas
# ORIGEN EDA : §11 · "isotónica en VAL; verificar con Brier y curva de fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              VAL son 750 pacientes; la isotónica puede sobreajustar. Si el Brier no mejora, decirlo.
# ══════════════════════════════════════════════════════════════════════════════
calibrators={}
for j,l in enumerate(LABELS):
    s=m_val[:,j]==1; yt=y_val[s,j]; yp=val_pred_raw[s,j]
    calibrators[l]=None if len(np.unique(yt))<2 else IsotonicRegression(out_of_bounds="clip").fit(yp,yt)
def apply_cal(P):
    O=P.copy()
    for j,l in enumerate(LABELS):
        if calibrators[l] is not None: O[:,j]=calibrators[l].predict(P[:,j])
    return O
oof_cal=apply_cal(oof_train); val_pred=apply_cal(val_pred_raw); test_pred=apply_cal(test_pred_raw)

# ── B2 · ¿mejora realmente la calibración? Brier antes vs después (medido en VAL) ─────────────
brier_pre,_       = calibration_report(val_pred_raw, y_val, m_val)
brier_post,curvas = calibration_report(val_pred,     y_val, m_val)
cal_cmp=brier_pre.merge(brier_post,on="etiqueta",suffixes=("_sin_calibrar","_calibrado"))
cal_cmp["mejora"]=cal_cmp["Brier_sin_calibrar"]-cal_cmp["Brier_calibrado"]
print("== B2 · Brier en VALIDACION (menor = mejor) ==")
print(cal_cmp.to_string(index=False,float_format=lambda v:f"{v:.4f}"))
print(f"   Mejora en {(cal_cmp['mejora']>0).sum()}/{len(cal_cmp)} etiquetas")
cal_cmp.to_csv(OUTPUT_DIR/"calibracion_brier.csv",index=False)

fig,ax=plt.subplots(figsize=(6.4,6)); ax.plot([0,1],[0,1],"--",color="gray",lw=1,label="calibracion perfecta")
for l in LABELS:
    fr,mp=curvas.get(l,(np.array([]),np.array([])))
    if len(fr): ax.plot(mp,fr,"o-",ms=4,lw=1.6,label=l)
ax.set_xlabel("probabilidad predicha media"); ax.set_ylabel("frecuencia observada")
ax.set_title("ECG v1 · Curva de fiabilidad tras calibracion (VAL)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG/"curva_fiabilidad_v1.png",dpi=150,bbox_inches="tight"); plt.show()

# ── B1 · Tres puntos de operación fijados en VAL ──────────────────────────────────────────────
PUNTOS=operating_points(val_pred,y_val,m_val,sens_target=0.90,spec_target=0.90)
thr_val=PUNTOS["f1"]
print("\n== B1 · Umbrales por punto de operacion (fijados en VAL) ==")
print("{:18s} {:>6} {:>18} {:>18}".format("Etiqueta","F1","Cribado(Se>=.90)","Confirm(Sp>=.90)"))
for l in LABELS:
    print("{:18s} {:6.2f} {:18.2f} {:18.2f}".format(l,PUNTOS["f1"][l],PUNTOS["cribado"][l],PUNTOS["confirm"][l]))

# ── B3 · PRE-REGISTRO de la regla de decisión (ANTES de tocar TEST) ───────────────────────────
json.dump({"modelo":"ECG v1 ligero (ResNet1D base32, senal 1000)",
           "etiquetado":"POS=(==1); NEG=(==0)|(NaN->0); -1 ENMASCARADO (U-Ignore); sin derivar de No Finding",
           "metrica_primaria":"AUC-PR (macro_AP_path); ROC secundaria; IC bootstrap 1000",
           "calibracion":"isotonica ajustada en VAL",
           "senal":"12 derivaciones x 5000 muestras normalizadas a [-1,1]; NO se re-normaliza (anexo EDA)",
           "puntos_operacion":{"f1":PUNTOS["f1"],"cribado_Se>=0.90":PUNTOS["cribado"],"confirmacion_Sp>=0.90":PUNTOS["confirm"]},
           "cxr_view":"EXCLUIDA como predictor (proxy de gravedad); solo para estratificar",
           "nota":"TEST se evalua UNA sola vez con estos umbrales; no se reajusta nada despues."},
          open(OUTPUT_DIR/"preregistro_regla_decision.json","w",encoding="utf-8"),indent=2,default=str,ensure_ascii=False)
print("\n== B3 · Regla PRE-REGISTRADA en preregistro_regla_decision.json (test aun sin tocar)")



In [ ]:
# CELDA 11 · EVALUACIÓN EN TEST (UNA sola vez, con la regla PRE-REGISTRADA)
# Contenido: discriminación con IC bootstrap (B4) · los 3 puntos de operación con Se/Sp/VPP/VPN (B1)
#            · consistencia de «Sin hallazgo» (B6) · estratificación por subgrupos y proyección (B8).

# ── Discriminación por etiqueta, con IC bootstrap de AP y de AUC (B4) ─────────────────────────
M      = multilabel_metrics(test_pred, y_test, m_test, thresholds=thr_val)
ci_ap  = bootstrap_ci_metric(test_pred, y_test, m_test, metric="ap")
ci_auc = bootstrap_ci_metric(test_pred, y_test, m_test, metric="auc")
rows=[]
print("== DISCRIMINACION (test) · AP es la metrica primaria; su linea base es la PREVALENCIA ==")
print("{:18s} {:>7} {:>16} {:>6} {:>7} {:>16} {:>5} {:>5}".format(
    "Etiqueta","AP","IC95% AP","prev","AUC","IC95% AUC","N+","N-"))
print("-"*96)
for l in LABELS:
    m=M[l]; auc=f"{m['AUC']:.4f}" if not np.isnan(m['AUC']) else "  N/A"
    la,ha=ci_ap[l]; lu,hu=ci_auc[l]
    fa=f"[{la:.3f},{ha:.3f}]" if not np.isnan(la) else "        N/A"
    fu=f"[{lu:.3f},{hu:.3f}]" if not np.isnan(lu) else "        N/A"
    print("{:18s} {:7.4f} {:>16} {:6.3f} {:>7} {:>16} {:5d} {:5d}".format(
        l,m["AP"],fa,m["prevalencia"],auc,fu,m["n_pos"],m["n_neg"]))
    rows.append({"label":l,**{k:m[k] for k in ["AUC","AP","F1","sens","spec","n_pos","n_neg","prevalencia"]},
                 "AP_ci_lo":la,"AP_ci_hi":ha,"AUC_ci_lo":lu,"AUC_ci_hi":hu,
                 "AP_supera_prevalencia":bool(m["AP"]>m["prevalencia"])})
print("-"*96)
mlo,mhi=ci_ap["macro_path"]; ulo,uhi=ci_auc["macro_path"]
print(f"MACRO patol.: AP={M['macro_AP_path']:.4f} IC95%=[{mlo:.3f},{mhi:.3f}]  (PRIMARIA)")
print(f"              AUC={M['macro_AUC_path']:.4f} IC95%=[{ulo:.3f},{uhi:.3f}]  (secundaria)")
n_ok=sum(r["AP_supera_prevalencia"] for r in rows if r["label"] in PATHOLOGY)
print(f"Patologias con AP > prevalencia (= senal real): {n_ok}/{len(PATHOLOGY)}")
pd.DataFrame(rows).to_csv(OUTPUT_DIR/"metrics_per_label_v1.csv",index=False)

# ── B1 · Métricas clínicas en los tres puntos de operación ────────────────────────────────────
# ORIGEN EDA §11: "Reportar Se/Sp/VPP/VPN". Cribar y confirmar son decisiones clinicas distintas.
clin=pd.concat([clinical_report(test_pred,y_test,m_test,PUNTOS[k],punto=nm)
                for k,nm in [("f1","f1"),("cribado","cribado_Se>=0.90"),("confirm","confirmacion_Sp>=0.90")]],
               ignore_index=True)
print("\n== PUNTOS DE OPERACION (test) ==")
cab="    {:18s} {:>5} {:>6} {:>6} {:>6} {:>6} {:>4} {:>4}".format("Etiqueta","thr","Se","Sp","VPP","VPN","FN","FP")
for punto in clin["punto"].unique():
    print("\n  · Punto "+str(punto)+":"); print(cab)
    for _,r in clin[clin["punto"]==punto].iterrows():
        print("    {:18s} {:5.2f} {:6.3f} {:6.3f} {:6.3f} {:6.3f} {:4d} {:4d}".format(
            r["etiqueta"],r["umbral"],r["Se"],r["Sp"],r["VPP"],r["VPN"],int(r["FN"]),int(r["FP"])))
clin.to_csv(OUTPUT_DIR/"puntos_operacion_test.csv",index=False)

# ── B6 · Coherencia interna de «Sin hallazgo» ─────────────────────────────────────────────────
cons=consistency_no_finding(test_pred)
print(f"\n== B6 · corr(P(Sin hallazgo), max P(patologia))={cons['corr_NoFinding_vs_maxPatologia']:.3f} "
      f"(debe ser NEGATIVA) · incoherentes={cons['pct_incoherentes']:.1f} %")

# ── B8 · Equidad y robustez por subgrupo (incluye la proyección radiográfica) ─────────────────
strat=stratified_report(df_test,test_pred,y_test,m_test)
print("\n== B8 · Rendimiento por subgrupo (n<60 => posible ruido, no inequidad) ==")
print(strat.to_string(index=False,float_format=lambda v:f"{v:.4f}"))
strat.to_csv(OUTPUT_DIR/"estratificacion_subgrupos.csv",index=False)

json.dump({"version":"ECG v1 ligero (ResNet1D base32, senal 1000)","primary_metric":"AUC-PR (macro_AP_path)",
           "cv_macro_path":float(np.nanmean(fold_macro)),
           "test_macro_ap_path":M["macro_AP_path"],"test_macro_ap_ci":[mlo,mhi],
           "test_macro_auc_path":M["macro_AUC_path"],"test_macro_auc_ci":[ulo,uhi],
           "test_per_label":{l:M[l] for l in LABELS},"consistencia_no_finding":cons},
          open(OUTPUT_DIR/"summary_v1.json","w",encoding="utf-8"),indent=2,default=str,ensure_ascii=False)
print("\nGuardados: metrics_per_label_v1.csv · puntos_operacion_test.csv · estratificacion_subgrupos.csv · summary_v1.json")



In [ ]:
# CELDA 12 · MATRICES DE CONFUSIÓN + AUC por etiqueta
fig,axes=plt.subplots(2,3,figsize=(13,8)); fig.suptitle("ECG v1 — Matrices de confusión (test)",fontweight="bold")
for j,l in enumerate(LABELS):
    ax=axes[j//3,j%3]; s=m_test[:,j]==1; yt=y_test[s,j]; yp=(test_pred[s,j]>=thr_val.get(l,0.5)).astype(int)
    if len(yt)==0: ax.axis("off"); continue
    sns.heatmap(confusion_matrix(yt,yp,labels=[0,1]),annot=True,fmt="d",cmap="Purples",cbar=False,ax=ax,
                xticklabels=["P0","P1"],yticklabels=["R0","R1"]); ax.set_title(f"{l}",fontsize=10)
plt.tight_layout(); plt.savefig(FIG/"confusion_v1.png",dpi=150,bbox_inches="tight"); plt.show()
fig2,ax=plt.subplots(figsize=(9,4.5)); aucs=[M[l]["AUC"] for l in LABELS]
ax.bar(range(N_LABELS),[0 if np.isnan(a) else a for a in aucs],color=["#c0392b" if (np.isnan(a) or a<0.55) else "#8e44ad" for a in aucs],alpha=0.85)
ax.axhline(0.5,color="gray",ls="--"); ax.set_xticks(range(N_LABELS)); ax.set_xticklabels([l[:11] for l in LABELS],rotation=30,ha="right")
ax.set_ylim(0,1); ax.set_ylabel("AUC (test)"); ax.set_title("ECG v1 · AUC por etiqueta")
plt.tight_layout(); plt.savefig(FIG/"auc_por_etiqueta_v1.png",dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
# CELDA 13 · EXPORTAR OOF/val/test PARA EL STACKING (hadm_id, ecg_<label>, ecg_<label>_cal)
def save_predictions(df, raw, cal, name):
    cols={"hadm_id":df["hadm_id"].to_numpy()}
    for j,l in enumerate(LABELS):
        key=l.replace(" ","_"); cols[f"ecg_{key}"]=raw[:,j]; cols[f"ecg_{key}_cal"]=cal[:,j]
    out=pd.DataFrame(cols); p=OUTPUT_DIR/f"ecg_pred_{name}.csv"; out.to_csv(p,index=False); print("   guardado",p.name)
save_predictions(df_train, oof_train, oof_cal, "oof_train")
save_predictions(df_val, val_pred_raw, val_pred, "val")
save_predictions(df_test, test_pred_raw, test_pred, "test")
print("OOF/val/test del ECG v1 exportados.")

---
## ✅ Resumen — ECG v1 (ligero)

**ResNet-1D base 32** sobre señal de 1000 puntos, con pérdida **BCE enmascarada** y `pos_weight` por
etiqueta, calibración isotónica ajustada en VAL y **OOF sin fuga** para el stacking multimodal.
Adopta el **etiquetado FINAL** (negativo = 0 + NaN→0; −1 enmascarado; sin derivar de *No Finding*),
la métrica **AUC-PR con IC bootstrap** y el **kit clínico B1, B2, B3, B4, B6 y B8**.

### 📌 Para la documentación posterior (recuadros naranjas a redactar con los resultados)
- **¿Alguna AP supera su prevalencia?** Es la prueba de que hay señal real. Si no la supera, esa
  etiqueta **no es aprendible desde el ECG**. El EDA ya anticipaba que el ECG aporta poco en solitario:
  su valor es como **modalidad complementaria** en la fusión.
- **Puntos de operación**: contrastar los **falsos negativos** del punto de cribado con los **falsos
  positivos** del de confirmación, y decir explícitamente si alguno es clínicamente asumible.
- **Calibración**: ¿mejora el Brier tras la isotónica? Recordar que hay que **recalibrar tras la fusión**
  (la calibración por modalidad no se conserva al combinar).
- **Consistencia**: la correlación entre P(*Sin hallazgo*) y max P(patología) debe ser **negativa**;
  si sale ~0 o positiva, la cabeza de normalidad no está aprendiendo el concepto.
- **Equidad**: varios subgrupos del test (464 pacientes) tienen n<60 ⇒ las diferencias pueden ser
  **ruido**, no inequidad. No afirmar sesgo sin intervalo de confianza.
- **Proyección AP/PA**: comentar si el rendimiento difiere entre proyecciones (sesgo de espectro).
- Comparar v1/v2/v3 **con la misma métrica y el mismo etiquetado**; las cifras antiguas no valen.

